# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [3]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [60]:
# Initialize and constants
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
OLLAMA_BASE_URL = "http://localhost:11434/v1"

load_dotenv(override=True)

google_api_key = os.getenv("GOOGLE_API_KEY")

api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")


MODEL = 'gpt-5-nano'
openai = OpenAI()
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
gModel = "gemini-2.5-flash"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
oMODEL = 'gpt-oss:120b-cloud'

API key looks good so far


In [37]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.co

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [38]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [39]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [40]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [66]:
def select_relevant_links(url):
    response = ollama.chat.completions.create(
        model=oMODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [67]:
select_relevant_links("https://edwarddonner.com")


{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'blog', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'curriculum', 'url': 'https://edwarddonner.com/curriculum/'}]}

In [49]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = gemini.chat.completions.create(
        model=gModel,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [50]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 7 relevant links


{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'product/service', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'product/service', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'product/service', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'product/service', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'related product/company',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'}]}

In [51]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 7 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/'},
  {'type': 'enterprise solutions page',
   'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'brand information page', 'url': 'https://huggingface.co/brand'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'learn page', 'url': 'https://huggingface.co/learn'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [52]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [53]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 17 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
Qwen/Qwen3.5-35B-A3B
Updated
4 days ago
•
681k
•
845
Qwen/Qwen3.5-27B
Updated
6 days ago
•
319k
•
538
unsloth/Qwen3.5-35B-A3B-GGUF
Updated
4 days ago
•
570k
•
468
Qwen/Qwen3.5-122B-A10B
Updated
1 day ago
•
150k
•
383
LiquidAI/LFM2-24B-A2B
Updated
3 days ago
•
11.9k
•
239
Browse 2M+ models
Spaces
Running
on
Zero
Featured
1.79k
Qwen Image Multiple Angles 3D Camera
🎥
1.79k
Change the camera angle of a photo with AI
Running
on
Zero
Featured
292
Omni Video Factory
🏆
292
text to video, image to video, video extend
Running
on
Zero
MCP
1.05k

In [54]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [55]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [57]:
get_brochure_user_prompt("bitspilaniDigital", "https://bitspilani-digital.edu.in/")

Selecting relevant links for https://bitspilani-digital.edu.in/ by calling gpt-5-nano
Found 20 relevant links


"\nYou are looking at a company called: bitspilaniDigital\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nUGC Approved Online Degree Courses in India | BITS Pilani Digital\n\nApplications open for May 2026 batch for BS, BSc & MSc programs | Certificate program: AI Engineering & MLOps | Last Date of Application: 12th March, 2026\n×\nDegrees\nBachelors in Data Science & AI\nBachelors in Computer Science\nMasters in Data Science & AI\n+91 8423662366\nCertificates\nAI Engineering and MLOps\nCybersecurity and Secure Software Development\n+91 8423662366\nAbout Us\nInfo Hub\nRegulatory Provision\nExam Centers\nBITS Pilani Website\nEducation Policy\nDEB-ID\nFAQs\nAI Engineering & MLOps\nCybersecurity and Secure\nSoftware Development\nMasters in Data Science & AI\n+91 8423662366\nBlogs\n+91 8423662366\nBITS Pilani Digital Buzz\nSocial Buzz\nPodcasts\nPR C

In [61]:
def create_brochure(company_name, url):
    response = ollama.chat.completions.create(
        model=oMODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [63]:
create_brochure("bitspilaniDigital", "https://bitspilani-digital.edu.in/")

Selecting relevant links for https://bitspilani-digital.edu.in/ by calling gpt-5-nano
Found 5 relevant links


**BITS Pilani Digital – Empowering the Next Generation of Technologists**  
*Excellence Made Yours*  

---

### Who We Are  
BITS Pilani Digital is the online learning arm of the iconic BITS Pilani university. Built as a **digital‑first ecosystem**, we combine the institute’s world‑class academic rigor with an empathetic, learner‑centric design. Our mission is to make premium education **accessible, flexible, and industry‑ready** for students across India and beyond.

---

### What We Offer  

| Category | Programs (May 2026 intake) | Highlights |
|----------|---------------------------|------------|
| **Undergraduate Degrees** | • Bachelors in Data Science & AI  <br>• Bachelors in Computer Science | Industry‑grade projects, mentorship from BITS faculty & industry practitioners |
| **Post‑Graduate Degrees** | • Masters in Data Science & AI | Research‑oriented curriculum, real‑world case studies |
| **Professional Certificates** | • AI Engineering & MLOps  <br>• Cybersecurity & Secure Software Development | Short‑duration, skill‑focused, stack‑ready for immediate job impact |

All programs are **UGC‑approved** and carry the global reputation of the BITS brand.

---

### Learning Experience  

- **Industry‑Grade Projects** – Work on the same challenges that leading tech firms solve.  
- **Cut‑Edge Tools** – Hands‑on training with the software and platforms professionals use daily.  
- **Faculty & Practitioners** – Learn from celebrated BITS professors and seasoned industry experts.  
- **Structured Extracurriculars** – Workshops, hackathons, real‑world challenges, and co‑curricular clubs.  
- **Learners Community** – Peer‑to‑peer support, discussion forums, and alumni networks keep you motivated.  
- **Campus Immersion** – Optional one‑year‑ly campus visits for a taste of the BITS campus life.  

---

### Culture & Values  

- **Student‑First:** Courses are designed with empathy, meeting learners where they are.  
- **Innovation‑Driven:** We stay ahead of technological trends, continuously updating curricula.  
- **Collaborative:** A supportive community where every voice matters.  
- **Lifelong Learning:** Resources stay available for alumni to upskill throughout their careers.  

---

### Who Benefits  

- **Aspiring Technologists** looking for a reputable, flexible degree or certificate.  
- **Working Professionals** seeking to upskill in AI, data science, or cybersecurity without leaving their jobs.  
- **Employers & Industry Partners** that need a pipeline of job‑ready talent trained on real‑world projects.  

---

### Career Opportunities  

While our primary focus is education, BITS Pilani Digital regularly hires for roles such as:  

- Academic & Industry Faculty (full‑time, adjunct)  
- Course Designers & Instructional Designers  
- Learning Experience & Community Managers  
- Technical Support & Platform Engineers  
- Marketing, Admissions, and Outreach specialists  

*Check our Careers page or contact the recruitment team for the latest openings.*

---

### How to Join  

- **Application Deadline:** 12 March 2026  
- **Start Date:** May 2026 (BS, BSc & MSc)  
- **Contact:** +91 84236 62366 or visit the **Apply Now** portal on the website.  

---

### Stay Connected  

- **Website:** [bits-pilani.edu.in/digital](https://bits-pilani.edu.in/digital)  
- **Blogs, Podcasts & Social Buzz:** Regular updates on industry trends, student stories, and campus life.  

---

> **“Excellence should feel like it’s made just for you.”** – BITS Pilani Digital  

Ready to shape the future? Join a community where ambition meets opportunity, and turn your potential into impact.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [68]:
def stream_brochure(company_name, url):
    stream = ollama.chat.completions.create(
        model=oMODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [71]:
stream_brochure("Bajaj Life Insurance", "https://www.bajajlifeinsurance.com")

# Bajaj Life Insurance  
*Your trusted partner for a secure financial future*

---

## 🌟 Who We Are  

Bajaj Life Insurance (formerly **Bajaj Allianz Life Insurance**) blends over two decades of insurance expertise with the forward‑thinking spirit of the Bajaj Group. Our mission is simple: **provide simple, affordable, and high‑quality life‑insurance solutions for every stage of life**—whether you’re a young professional, a family‑focused parent, a senior citizen, or an NRI living abroad.

- **Customer‑first mindset** – products are designed around real life goals.  
- **Inclusive coverage** – special plans for women, seniors, diabetics, and NRIs.  
- **Digital‑ready** – buy, manage, and claim your policy entirely online.  
- **Strong heritage** – built on the legacy of Bajaj Allianz, now operating as an independent, purpose‑driven insurer.

---

## 🎯 Why Choose Bajaj Life Insurance?  

| Feature | Benefit |
|---------|----------|
| **0% GST on Term Plans** | Lower upfront cost, more savings. |
| **Up‑to 16% discount on 1st‑year premium** | Immediate value on the eTouch II term plan. |
| **Free health cover up to ₹31,000 p.a.** | Added protection without extra premium. |
| **Lifelong income for parents** | Peace of mind for your family’s future. |
| **Tax deductions up to ₹46,800** | Maximise savings under prevailing tax laws. |
| **High claim‑settlement ratio** | Trusted and reliable payouts when you need them. |
| **Online term‑insurance calculator** | Instant quote and coverage comparison. |
| **Tailored NRI & senior‑citizen plans** | Global reach, local relevance. |

---

## 📦 Our Product Portfolio  

### 1. Term Insurance  
- **eTouch II** – Non‑linked, non‑participating, up to 16 % first‑year discount.  
- **Superwoman Term** – Exclusively for women, with added health benefits.  
- **iSecure II** – Diabetic‑friendly term plan (HbA1c < 8).  
- Coverage options from **₹25 Lakh** to **₹5 Crore** and terms of **15–30 years**.

### 2. Investment‑Linked Plans (ULIPs)  
- **Goal Assure IV**, **Smart Wealth Goal V**, **Supreme Gold** – market‑linked growth with flexible fund choices (New Fund @ ₹10 NAV).  

### 3. Savings & Endowment Plans  
- **ACE**, **Assured Wealth Goal Platinum**, **Guaranteed Wealth Goal**, **Goal Suraksha** – steady wealth accumulation with guaranteed returns.

### 4. Child & Retirement Solutions  
- **Child Insurance Plans** – secure your child’s future education & milestones.  
- **Smart Pension**, **Guaranteed Pension Goal II**, **LongLife Goal III** – systematic retirement income.

### 5. NRI Specific Plans  
- Term, ULIP, and investment options designed for NRIs (including UAE) with tax benefits and flexible currency options.

---

## 👥 Who We Serve  

- **Young Professionals & Families** – affordable term cover, savings, and child plans.  
- **Women & Seniors** – dedicated products like Superwoman Term and senior‑citizen term plans.  
- **Diabetics & High‑Risk Groups** – specialized underwriting (iSecure II).  
- **NRIs & Global Citizens** – cross‑border coverage with UAE‑focused term options.  

Our policies are built to adapt to life’s changing needs, delivering protection and wealth creation for **individuals, families, and expatriate communities**.

---

## 📱 Digital Experience  

- **Buy online in minutes** – fully digital onboarding, e‑KYC, and instant policy issuance.  
- **Term Insurance Calculator** – instant premium & coverage estimate.  
- **Claim Settlement Portal** – track claims 24/7, backed by a high settlement ratio.  
- **Mobile App & Customer Dashboard** – view policies, make premium payments, and update details on the go.

---

## 🏦 Financial Strength & Trust  

- Backed by the **Bajaj Group**’s robust financial foundation.  
- Consistently high **claim settlement ratio** (industry‑leading).  
- Transparent pricing with **0 % GST** on term plans and clear policy documents.

---

## 🌱 Culture & Values  

- **Inclusivity** – product suites for every demographic, reflecting our belief that insurance is for all.  
- **Innovation** – continuous investment in digital tools, AI‑driven underwriting, and customer service platforms.  
- **Integrity** – ethical practices, clear communication, and a customer‑first policy approach.  
- **Sustainability** – responsible investing, community outreach, and financial literacy programs.

---

## 🚀 Join Our Team  

Bajaj Life Insurance is always looking for passionate professionals who share our vision of **making life insurance simple, accessible, and rewarding**.  

- **Career paths**: underwriting, actuarial, digital product development, customer service, sales & marketing, finance & risk.  
- **Work environment**: collaborative, merit‑based, and supportive of continuous learning.  

📧 **Email:** careers@bajallife.com  
🌐 **Explore opportunities:** [Bajaj Life Careers](https://www.bajallife.com/careers)

---

## 📞 Get in Touch  

- **Website:** https://www.bajallife.com  
- **Customer Care:** 1800 202 2020 (toll‑free)  
- **Email:** contact@bajallife.com  
- **Social:** Follow us on LinkedIn, Twitter, and Facebook for the latest updates and financial tips.

---

**Bajaj Life Insurance – Secure today, prosper tomorrow.**

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>